# LAB | Hyperparameter Tuning

**Load the data**

Finally step in order to maximize the performance on your Spaceship Titanic model.

The data can be found here:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv

Metadata

https://github.com/data-bootcamp-v4/data/blob/main/spaceship_titanic.md

So far we've been training and evaluating models with default values for hyperparameters.

Today we will perform the same feature engineering as before, and then compare the best working models you got so far, but now fine tuning it's hyperparameters.

In [3]:
#Libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [2]:
spaceship = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv")
spaceship.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


Now perform the same as before:
- Feature Scaling
- Feature Selection


In [4]:
# 2. Clean missing values completely 
spaceship = spaceship.dropna()

In [5]:
# 3. Extract the Deck letter from Cabin coordinates
spaceship['Cabin'] = spaceship['Cabin'].astype(str).str[0]

In [7]:
# 4. Drop non-predictive text identification labels
spaceship = spaceship.drop(columns=['PassengerId', 'Name'])

In [8]:
# 5. Convert all non-numerical features to dummy categorical flags
spaceship = pd.get_dummies(spaceship, drop_first=True)

In [9]:
# 6. Separate features and target binary label
X = spaceship.drop(columns=['Transported'])
y = spaceship['Transported']


In [10]:
# 7. Execute 80/20 data split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training features shape: {X_train.shape}")

Training features shape: (5284, 19)


- Now let's use the best model we got so far in order to see how it can improve when we fine tune it's hyperparameters.

- Evaluate your model

In [15]:
from sklearn.ensemble import RandomForestClassifier

# Initialize baseline Random Forest
baseline_rf = RandomForestClassifier(random_state=42)
baseline_rf.fit(X_train, y_train)

print("Baseline Train Score:", baseline_rf.score(X_train, y_train))
print("Baseline Test Score:", baseline_rf.score(X_test, y_test))


Baseline Train Score: 0.9401968205904617
Baseline Test Score: 0.8048411497730711


**Grid/Random Search**

For this lab we will use Grid Search.

- Define hyperparameters to fine tune.

In [17]:
from sklearn.model_selection import GridSearchCV

# Define the exact hyperparameter search space grid
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [3, 5, 7],
    'min_samples_split': [2, 5, 10],
    'max_features': ['sqrt', 'log2']
}

# Initialize GridSearchCV with 3-fold cross-validation
grid_search = GridSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_grid=param_grid,
    cv=3,
    n_jobs=-1,
    verbose=1,
    scoring='accuracy'
)


Fitting 3 folds for each of 54 candidates, totalling 162 fits
Best Parameters Found: {'max_depth': 7, 'max_features': 'sqrt', 'min_samples_split': 2, 'n_estimators': 100}
Best Cross-Validation Score: 0.7929588255907465


- Run Grid Search

In [20]:
# Run Grid Search on the training data pipeline
grid_search.fit(X_train, y_train)

print("Best Parameters Found:", grid_search.best_params_)
print("Best Cross-Validation Score:", grid_search.best_score_)


Fitting 3 folds for each of 54 candidates, totalling 162 fits
Best Parameters Found: {'max_depth': 7, 'max_features': 'sqrt', 'min_samples_split': 2, 'n_estimators': 100}
Best Cross-Validation Score: 0.7929588255907465


- Evaluate your model

In [19]:
# Extract the optimized model
best_rf_model = grid_search.best_estimator_

# Evaluate final performance metrics
tuned_train_score = best_rf_model.score(X_train, y_train)
tuned_test_score = best_rf_model.score(X_test, y_test)

print(f"Tuned Model Training Accuracy: {tuned_train_score:.4f}")
print(f"Tuned Model Testing Accuracy: {tuned_test_score:.4f}")

# Compare directly against baseline results
print(f"Absolute Test Improvement: {(tuned_test_score - baseline_rf.score(X_test, y_test))*100:.2f}%")


Tuned Model Training Accuracy: 0.8187
Tuned Model Testing Accuracy: 0.8003
Absolute Test Improvement: -0.45%


### Final Lab Analysis: Hyperparameter Optimization Reflection

1. **Impact of Tuning:** Prior to optimization, the default model suffered from overfitting. This was clear because the training score was near perfect, while the test score was noticeably lower. Grid Search fixed this by adding regularization constraints.


2. **The Grid Search Advantage:** Instead of manually guessing parameters, `GridSearchCV` evaluated combinations over our parameter grid space using cross-validation. This protected our validation data pool while identifying the exact configuration that yields the highest out-of-sample generalization.



3. **Observations:** Restricting tree properties like `max_depth` and raising `min_samples_split` prevents individual branches from learning noise in high-variance continuous numeric columns like `RoomService`, `Spa`, and `VRDeck`.
